## Testing on-the-fly using trained DDQNs

This notebook contains the workflow for testing trained DDQNs across all problem settings.

The agent solves a **single-dimensional discrete action space**: for every state `[n, f(x)]` it
picks one index into the portfolio of $\lambda$ values declared by `action_choices` in the config.
The greedy action of the Q-network over all fitness values `f(x)` in `[0, n)` therefore *is* the
policy, which we then run with the (1+($\lambda$, $\lambda$))-GA to measure the ERT.

### Import the necessary packages and developed modules

**Note**: the notebook must be able to import the `onemax_dac` and `dacbench` packages from the
project root. Either start Jupyter from the project root, or keep the `sys.path` line below.

In [6]:
import sys

sys.path.append("..")

import numpy as np
import torch
from joblib import Parallel, delayed
from tqdm import tqdm

from onemax_dac.train_ddqn import QNetwork
from onemax_dac.utils import read_config, make_env
from onemax_dac.evals import ollga_single_run

### Example Testing Configuration
- problem_size: 500
- state_dim: 2 — the observation is `n, f(x)`
- net_arch: [50, 50]
- n_eval_episodes: 1000
- num_workers: 4

The config file drives everything else (portfolio of $\lambda$ values, instance set, reward
choice), so pick the one matching the checkpoint. `best_model_shifting_n500.pt` was trained with
`onemax_n500_ddqn_as.yml` (`reward_choice: imp_minus_evals_shifting`). For another problem size,
change `CONFIG_FILE` and `CKPT_FILE` together.

In [7]:
CONFIG_FILE = "../onemax_dac/configs/onemax_n500_ddqn_as.yml"
CKPT_FILE = "../resources/ddqn_ckpts/best_model_shifting_n500.pt"
N_EVAL_EPISODES = 1000
NUM_WORKERS = 4

(
    exp_params,
    bench_params,
    agent_params,
    train_env_params,
    eval_env_params,
) = read_config(CONFIG_FILE)

env = make_env(bench_params, eval_env_params)
inst_id = env.instance_id_list[0]
n = env.instance_set[inst_id]["size"]

# the portfolio of lambda values the agent selects from
action_choices = env.action_choices[inst_id][0]
state_dim = env.observation_space.shape[0]

print(f"problem size : {n}")
print(f"state dim    : {state_dim}")
print(f"action space : {env.action_space} -> {list(action_choices)}")

problem size : 500
state dim    : 2
action space : Discrete(9) -> [1, 2, 4, 8, 16, 32, 64, 128, 256]


### Load the trained checkpoint

In [8]:
q_net = QNetwork(
    state_dim=state_dim,
    n_actions=len(action_choices),
    net_arch=agent_params["net_arch"],
    use_dueling=False,
)
q_net.load_state_dict(torch.load(CKPT_FILE, map_location="cpu", weights_only=True))
q_net.eval()

QNetwork(
  (fc1): Linear(in_features=2, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=50, bias=True)
  (fc3): Linear(in_features=50, out_features=9, bias=True)
)

### Read out the learned policy

One greedy action per fitness value: `policy[f_x]` is the $\lambda$ to use in state `f(x) = f_x`.

In [9]:
states = torch.tensor([[n, f_x] for f_x in range(n)], dtype=torch.float32)
with torch.no_grad():
    action_indices = q_net(states).argmax(dim=1).numpy()

policy = [int(action_choices[idx]) for idx in action_indices]
print(f"policy[f(x)=n-10..n-1]: {policy[-10:]}")

policy[f(x)=n-10..n-1]: [16, 16, 16, 16, 16, 16, 16, 16, 16, 32]


### Run test and observe the ERT

Each run executes the (1+($\lambda$, $\lambda$))-GA until the optimum is found (or the `0.8 * n^2`
cutoff is hit) and returns the number of solution evaluations. The ERT is the mean over
`N_EVAL_EPISODES` runs.

In [10]:
runtimes = Parallel(n_jobs=NUM_WORKERS)(
    delayed(ollga_single_run)(bench_params, eval_env_params, policy, seed)
    for seed in tqdm(range(N_EVAL_EPISODES), desc="Parallel Progress")
)
runtimes = np.array(runtimes)
print(f"Runtime: {runtimes.mean():.2f} ± {runtimes.std():.2f}")

Parallel Progress: 100%|██████████| 1000/1000 [00:35<00:00, 28.08it/s]


Runtime: 3003.11 ± 330.69
